In [ ]:
import numpy as np
import mne
import os
from pactools import Comodulogram
from tqdm import tqdm
import warnings
import scipy.io as sio

warnings.filterwarnings('ignore')

# ====================== 1. Global Parameters ======================
SFREQ = 500

PHASE_FREQS = np.linspace(0.5, 20, 39)
AMP_FREQS = np.linspace(30, 90, 60)

N_SURROGATES = 200

# ====================== 2. Channel Definitions ======================
# Anterior channels (non-occipital)
ANTERIOR_CHANNELS = [
    'Fpz','Fp2','Fz','F4','F8',
    'FC1','FC2','FC6',
    'M1','C3','Cz','C4','T8','M2',
    'CP5','CP1','CP2',
    'P7','P3','Pz','P4','P8','POz'
]

# Occipital channels
OCCIPITAL_CHANNELS = ['O1', 'Oz', 'O2']

# All channels to load
ALL_CHANNELS = list(set(ANTERIOR_CHANNELS + OCCIPITAL_CHANNELS))

# ====================== 3. Bidirectional PAC Configuration ======================
PAC_DIRECTIONS = {
    'top_down': {
        'phase': ANTERIOR_CHANNELS,
        'amp': OCCIPITAL_CHANNELS
    },
    'bottom_up': {
        'phase': OCCIPITAL_CHANNELS,
        'amp': ANTERIOR_CHANNELS
    }
}

# ====================== 4. Data Paths ======================
GROUPS = {
    '10hz': {
        'subjects': [
           "100306", "100412", "100515", "100723", "100927", "101029", "101139","101449", 
                     "101551", "101656", "101758", "101861", "101965", "102274", "102375", "102478", "102580"
        ],
        'source_dir': r'D:\山东第一医科大学\数据\TI_TASK_DATA\Task2\八因子\预处理\10HZ'
    }
}

OUT_BASE_DIR = r'D:\山东第一医科大学\数据\Task\八因子\频段耦合\跨被试\pac\其他调制枕叶'

TIME_POINTS = ['post']
EMOTIONS = ['sad']

# ====================== 5. Single-Subject PAC Calculation ======================
def calculate_pac_for_subject(subject, source_dir, out_dir):
    """
    Compute cross-channel bidirectional PAC for one subject
    """
    factor_results = []

    for factor in range(1, 9):
        fif_path = os.path.join(
            source_dir, TIME_POINTS[0], EMOTIONS[0],
            subject, f'factor_{factor}.fif'
        )

        raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)

        data = raw.get_data(picks=ALL_CHANNELS)
        data = mne.filter.detrend(data, axis=1)
        data -= data.mean(axis=1, keepdims=True)

        ch_to_idx = {ch: i for i, ch in enumerate(ALL_CHANNELS)}
        factor_dir_results = {}

        # ====================== Bidirectional PAC ======================
        for direction, cfg in PAC_DIRECTIONS.items():
            phase_chs = cfg['phase']
            amp_chs = cfg['amp']

            n_phase = len(phase_chs)
            n_amp = len(amp_chs)

            cross_pac = np.zeros((n_phase, n_amp, len(AMP_FREQS), len(PHASE_FREQS)))
            cross_z = np.zeros_like(cross_pac)
            cross_m = np.zeros((n_phase, n_amp, N_SURROGATES))

            comod = Comodulogram(
                fs=SFREQ,
                low_fq_range=PHASE_FREQS,
                high_fq_range=AMP_FREQS,
                method='tort',
                n_surrogates=N_SURROGATES,
                progress_bar=False,
                n_jobs=5
            )

            for i_p, ch_p in enumerate(phase_chs):
                for i_a, ch_a in enumerate(amp_chs):
                    print(
                        f'Subject {subject} | Factor {factor} | '
                        f'{direction} | Phase: {ch_p} → Amp: {ch_a}'
                    )

                    comod.fit(
                        data[ch_to_idx[ch_p]],
                        data[ch_to_idx[ch_a]]
                    )

                    cross_pac[i_p, i_a] = comod.comod_.T
                    cross_z[i_p, i_a] = comod.comod_z_score_.T
                    cross_m[i_p, i_a] = comod.surrogate_max_

            factor_dir_results[direction] = {
                'pac': cross_pac,
                'pac_z': cross_z,
                'surrogate_max': cross_m,
                'phase_channels': phase_chs,
                'amp_channels': amp_chs
            }

        factor_results.append(factor_dir_results)

    # ====================== Save Results ======================
    save_dict = {
        'results': factor_results,
        'freqs': {
            'phase_freqs': PHASE_FREQS,
            'amp_freqs': AMP_FREQS
        }
    }

    out_path = os.path.join(out_dir, 'pac_bidir.mat')
    sio.savemat(out_path, save_dict)
    print(f'Subject {subject} bidirectional PAC saved: {out_path}')

# ====================== 6. Batch Processing ======================
if __name__ == '__main__':

    for group_name, group_info in GROUPS.items():
        src_dir = group_info['source_dir']
        subjects = group_info['subjects']

        for time_point in TIME_POINTS:
            for emotion in EMOTIONS:
                for subject in tqdm(subjects, desc=f'{group_name}-{time_point}-{emotion}'):

                    out_dir = os.path.join(
                        OUT_BASE_DIR, group_name, time_point, emotion, subject
                    )
                    os.makedirs(out_dir, exist_ok=True)

                    calculate_pac_for_subject(
                        subject=subject,
                        source_dir=src_dir,
                        out_dir=out_dir
                    )

    print('✅ All subjects: bidirectional PAC computation completed')

In [ ]:
# ===================== Save Completion Message =====================
print(outpath, "saved successfully")
print(f"Raw PAC array shape: {np.array(factor_all_pac, dtype=object).shape}")
print(f"PAC Z-score array shape: {np.array(factor_all_z, dtype=object).shape}")
print(f"Surrogate max array shape: {np.array(factor_all_m, dtype=object).shape}")

In [ ]:
import numpy as np
import mne
import os
from pactools import Comodulogram
from tqdm import tqdm
import warnings
import scipy.io as sio

time_points = ['pre', 'post']
emotions = ['sad']

phase_freqs = np.linspace(0.5, 20, 39)
amp_freqs = np.linspace(30, 90, 60)

groups = {
    # '10hz': {
    #    'subjects': ["100306", "100412", "100515", "100723", "100927", "101029", "101139","101449",
    #                 "101551", "101656", "101758", "101861", "101965", "102274", "102375", "102478", "102580"],
    #    'source_dir': r'D:\山东第一医科大学\数据\Task\八因子\频段耦合\跨通道\pac\其他调制枕叶\10hz'
    'sham': {
        'subjects': ["300207", "300308", "300409", "300618", "300720", "300822", "300925", "301041",
                     "301250", "301354", "301455", "301560", "301662", "301766", "301867", "302177",
                     "302282", "302384"],
        'source_dir': r'D:\山东第一医科大学\数据\Task\八因子\频段耦合\跨通道\pac\其他调制枕叶\sham'
    }
}

out_dir = r'D:\山东第一医科大学\数据\Task\八因子\频段耦合\跨通道\pac\其他调制枕叶'

# ---------------------- Core Extraction Logic ----------------------
def extract_pac_from_results(results):
    """
    Fix empty dimensions and ensure correct PAC data shape
    """
    extracted_data = {'pac': [], 'pac_z': [], 'surrogate_max': []}

    # Flatten to 1D
    results_1d = np.ravel(results)

    # Iterate over 8 factors
    for factor_idx in range(min(8, len(results_1d))):
        try:
            factor_data = results_1d[factor_idx]

            # Extract top_down and bottom_up
            for direction in ['top_down', 'bottom_up']:
                # Convert MATLAB struct to dict
                dir_data = factor_data[direction]
                if isinstance(dir_data, np.void):
                    dir_dict = dir_data.item()
                else:
                    dir_dict = dir_data

                # Remove singleton dimensions
                pac = np.squeeze(dir_dict['pac'])
                pac_z = np.squeeze(dir_dict['pac_z'])
                surrogate_m = np.squeeze(dir_dict['surrogate_max'])

                # Append to list
                extracted_data['pac'].append(pac)
                extracted_data['pac_z'].append(pac_z)
                extracted_data['surrogate_max'].append(surrogate_m)

                # Debug print
                print(f"✅ Factor {factor_idx + 1}-{direction} PAC shape: {pac.shape}")

        except Exception as e:
            print(f"⚠️ Factor {factor_idx + 1} extraction failed: {str(e)[:50]}..., skipped")
            continue

    # Convert to numpy arrays (filter empty data)
    extracted_data['pac'] = np.array([x for x in extracted_data['pac'] if x.size > 0])
    extracted_data['pac_z'] = np.array([x for x in extracted_data['pac_z'] if x.size > 0])
    extracted_data['surrogate_max'] = np.array([x for x in extracted_data['surrogate_max'] if x.size > 0])

    return extracted_data

sfreq = 500

for group_name, info in tqdm(groups.items(), desc="Processing Groups"):
    source_dir, subjects = info['source_dir'], info['subjects']
    t_pac = []
    t_z = []
    t_m = []

    for time in tqdm(time_points, desc=f"{group_name}-TimePoints", leave=False):
        emo_pac = []
        emo_z = []
        emo_m = []

        for emotion in tqdm(emotions, desc=f"{time}-Emotion", leave=False):
            sub_pac = []
            sub_z = []
            sub_m = []

            for sub in tqdm(subjects, desc="Subjects", leave=False):
                mat_path = os.path.join(source_dir, time, emotion, sub, "pac_bidir.mat")
                print(f"Processing file: {mat_path}")

                # Load MAT file
                data_mat = sio.loadmat(mat_path, squeeze_me=True)
                results = data_mat['results']

                # Extract PAC data
                pac_data = extract_pac_from_results(results)

                # Assign data
                pac_data = pac_data['pac']
                z_data = pac_data['pac_z']
                m_data = pac_data['surrogate_max']

                print(pac_data.shape)

                sub_pac.append(pac_data)
                sub_z.append(z_data)
                sub_m.append(m_data)

        t_pac.append(sub_pac)
        t_z.append(sub_z)
        t_m.append(sub_m)

    # Save final combined data
    save_dict = {
        'pac': t_pac,
        'pac_z_score': t_z,
        'surrogate_max': t_m
    }

    outpath = os.path.join(out_dir, f'10hz_pac_z_m.mat')
    sio.savemat(outpath, save_dict)

    # ===================== Save Completion Message =====================
    print(outpath, "saved successfully")
    print(f"Raw PAC array shape: {np.array(t_pac, dtype=object).shape}")
    print(f"PAC Z-score array shape: {np.array(t_z, dtype=object).shape}")
    print(f"Surrogate max array shape: {np.array(t_m, dtype=object).shape}")